In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://www.ndb.int/projects/all-projects/page/{}/"

# Set headers to mimic a real browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)"
                  " Chrome/115.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
    "Referer": "https://www.ndb.int/"
}

def scrape_projects_from_page(page_num):
    url = BASE_URL.format(page_num)
    print(f"Scraping page {page_num}: {url}")
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    project_cards = soup.find_all("div", class_="project-card card card-transition")
    if not project_cards:
        print("No projects found on this page.")
        return []

    projects = []
    for card in project_cards:
        sector_div = card.find("div", class_="project-card-cat")
        sector = sector_div.text.strip() if sector_div else ""

        country_div = card.find("div", class_="project-card-country")
        country = country_div.text.strip() if country_div else ""

        title_div = card.find("div", class_="project-card-title")
        a_tag = title_div.find("a") if title_div else None
        project_name = a_tag.text.strip() if a_tag else ""
        project_url = a_tag['href'] if a_tag else ""

        type_div = card.find("div", class_="project-card-type")
        project_type = type_div.text.strip() if type_div else ""

        date_div = card.find("div", class_="project-card-date")
        date = date_div.text.strip() if date_div else ""

        projects.append({
            "Project Name": project_name,
            "Project URL": project_url,
            "Country": country,
            "Sector": sector,
            "Project Type": project_type,
            "Date": date
        })
    return projects

def scrape_all_projects(start_page=1, end_page=12, delay=1):
    all_projects = []
    for page_num in range(start_page, end_page + 1):
        projects = scrape_projects_from_page(page_num)
        if not projects:
            print(f"No projects on page {page_num}, stopping early.")
            break
        all_projects.extend(projects)
        time.sleep(delay)
    return all_projects

if __name__ == "__main__":
    projects = scrape_all_projects()
    df = pd.DataFrame(projects)
    df.to_excel("ndb_all_projects_cards2.xlsx", index=False)
    print(f"Scraped {len(projects)} projects total and saved to 'ndb_all_projects_cards2.xlsx'")



Scraping page 1: https://www.ndb.int/projects/all-projects/page/1/
Scraping page 2: https://www.ndb.int/projects/all-projects/page/2/
Scraping page 3: https://www.ndb.int/projects/all-projects/page/3/
Scraping page 4: https://www.ndb.int/projects/all-projects/page/4/
Scraping page 5: https://www.ndb.int/projects/all-projects/page/5/
Scraping page 6: https://www.ndb.int/projects/all-projects/page/6/
Scraping page 7: https://www.ndb.int/projects/all-projects/page/7/
Scraping page 8: https://www.ndb.int/projects/all-projects/page/8/
Scraping page 9: https://www.ndb.int/projects/all-projects/page/9/
Scraping page 10: https://www.ndb.int/projects/all-projects/page/10/
Scraping page 11: https://www.ndb.int/projects/all-projects/page/11/
Scraping page 12: https://www.ndb.int/projects/all-projects/page/12/
Scraped 142 projects total and saved to 'ndb_all_projects_cards2.xlsx'


In [8]:
!pip install fitz

   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   ---------------------------------------- 3.3/3.3 MB 21.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 23.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   --------------------------------------- 565.1/565.1 kB 18.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 5.0/5.0 MB 25.4 MB/s eta 0:00:00


In [10]:
!pip install frontend

In [15]:
import os

print("Current working directory:", os.getcwd())

static_paths = [
    './static',
    os.path.join(os.getcwd(), 'static'),
]

for path in static_paths:
    if not os.path.exists(path):
        print(f"Creating directory: {path}")
        os.makedirs(path, exist_ok=True)
    else:
        print(f"Directory exists: {path}")


Current working directory: C:\Users\Jaydeb
Creating directory: ./static
Directory exists: C:\Users\Jaydeb\static


In [17]:
!pip install tools

In [ ]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
import fitz  # PyMuPDF

# Ensure static folder exists (should be fine now)
os.makedirs('static', exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/115.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
    "Referer": "https://www.ndb.int/"
}

def get_pdf_link(project_url):
    try:
        response = requests.get(project_url, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        link = soup.find("a", class_="btn primary card-link download-icon external")
        if link and 'href' in link.attrs:
            return link['href']
    except Exception as e:
        print(f"Error fetching PDF link from {project_url}: {e}")
    return None

def extract_pdf_text(pdf_url):
    try:
        response = requests.get(pdf_url, headers=HEADERS)
        response.raise_for_status()
        pdf_data = response.content
        doc = fitz.open(stream=pdf_data, filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except Exception as e:
        print(f"Error extracting text from PDF {pdf_url}: {e}")
        return ""

def main():
    # Change this to the actual path of your Excel file
    input_excel_path = r'C:\Users\Jaydeb\Downloads\ndb_all_projects_cards2.xlsx'  
    output_csv_path = r'C:\Users\Jaydeb\Downloads\ndb_projects_with_pdf_text.csv'

    df = pd.read_excel(input_excel_path)

    if 'Project URL' not in df.columns:
        print("Error: 'Project URL' column not found in Excel.")
        return

    pdf_urls = []
    pdf_text_snippets = []

    for i, project_url in enumerate(df['Project URL'], 1):
        print(f"[{i}/{len(df)}] Processing: {project_url}")
        pdf_link = get_pdf_link(project_url)
        pdf_urls.append(pdf_link if pdf_link else "")

        pdf_text = ""
        if pdf_link:
            pdf_text = extract_pdf_text(pdf_link)
        pdf_text_snippets.append(pdf_text[:1000] if pdf_text else "")

        time.sleep(1)  # polite delay between requests

    df['PDF URL'] = pdf_urls
    df['PDF Text Snippet'] = pdf_text_snippets

    df.to_csv(output_csv_path, index=False)
    print(f"Saved updated data to {output_csv_path}")

if __name__ == "__main__":
    main()


[1/142] Processing: https://www.ndb.int/project/serentica-captive-renewable-energy-project/
Error extracting text from PDF https://www.ndb.int/wp-content/uploads/2025/06/Proposed-Projects_Summary_Public-Disclosure_Serentica.pdf: module 'fitz' has no attribute 'open'
[2/142] Processing: https://www.ndb.int/project/piramal-finance-affordable-housing-project/
Error extracting text from PDF https://www.ndb.int/wp-content/uploads/2025/05/Concept-Projects_PFL_Summary.pdf: module 'fitz' has no attribute 'open'
[3/142] Processing: https://www.ndb.int/project/graca-aranha-silvania-energy-transmission-project/
Error extracting text from PDF https://www.ndb.int/wp-content/uploads/2025/05/Proposed-Projects_Summary-for-Public-Disclosure_GATE-Concept-note.pdf: module 'fitz' has no attribute 'open'
[4/142] Processing: https://www.ndb.int/project/the-city-bank-sustainable-infrastructure-project/
Error extracting text from PDF https://www.ndb.int/wp-content/uploads/2024/04/CBL_Summary-for-Public-Disclo

In [1]:
!pip uninstall fitz


^C
Found existing installation: fitz 0.0.1.dev2
Uninstalling fitz-0.0.1.dev2:
  Would remove:
    c:\users\jaydeb\anaconda3\lib\site-packages\.ds_store
    c:\users\jaydeb\anaconda3\lib\site-packages\fitz-0.0.1.dev2.dist-info\*
    c:\users\jaydeb\anaconda3\lib\site-packages\fitz\*
    c:\users\jaydeb\anaconda3\lib\site-packages\scripts\*
    c:\users\jaydeb\anaconda3\scripts\fitz
    c:\users\jaydeb\anaconda3\scripts\log2design.py
Proceed (Y/n)? 


In [1]:
pip install pymupdf


  Using cached pymupdf-1.26.1-cp39-abi3-win_amd64.whl.metadata (3.4 kB)
Using cached pymupdf-1.26.1-cp39-abi3-win_amd64.whl (18.5 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import fitz
print(fitz.__doc__)
print(dir(fitz))


PyMuPDF 1.26.1: Python bindings for the MuPDF 1.26.2 library (rebased implementation).
Python 3.12 running on win32 (64-bit).

['ASSERT_PDF', 'Annot', 'AnyType', 'Archive', 'Base14_fontdict', 'Base14_fontnames', 'ByteString', 'CS_CMYK', 'CS_GRAY', 'CS_RGB', 'CheckColor', 'CheckFont', 'CheckFontInfo', 'CheckMarkerArg', 'CheckMorph', 'CheckParent', 'CheckQuad', 'CheckRect', 'ColorCode', 'Colorspace', 'ConversionHeader', 'ConversionTrailer', 'DeviceWrapper', 'DisplayList', 'Document', 'DocumentWriter', 'EMPTY_IRECT', 'EMPTY_QUAD', 'EMPTY_RECT', 'ENSURE_OPERATION', 'EPSILON', 'ElementPosition', 'EmptyFileError', 'FLT_EPSILON', 'FZ_MAX_INF_RECT', 'FZ_MIN_INF_RECT', 'FZ_RECOMPRESS_FAX', 'FZ_RECOMPRESS_J2K', 'FZ_RECOMPRESS_JPEG', 'FZ_RECOMPRESS_LOSSLESS', 'FZ_RECOMPRESS_NEVER', 'FZ_RECOMPRESS_SAME', 'FZ_SUBSAMPLE_AVERAGE', 'FZ_SUBSAMPLE_BICUBIC', 'FileDataError', 'FileNotFoundError', 'FitzDeprecation', 'Font', 'Graftmap', 'INFINITE_IRECT', 'INFINITE_QUAD', 'INFINITE_RECT', 'INVALID_NAME_CHARS

In [5]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
import fitz  # PyMuPDF

# Ensure static folder exists (avoid any related errors)
os.makedirs('static', exist_ok=True)

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
    "Referer": "https://www.ndb.int/"
}

def get_pdf_link(project_url):
    try:
        response = requests.get(project_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        link = soup.find("a", class_="btn primary card-link download-icon external")
        if link and 'href' in link.attrs:
            return link['href']
        else:
            print(f"No PDF link found on {project_url}")
    except Exception as e:
        print(f"Error fetching PDF link from {project_url}: {e}")
    return None

def extract_pdf_text(pdf_url):
    try:
        response = requests.get(pdf_url, headers=HEADERS, timeout=20)
        response.raise_for_status()
        pdf_data = response.content
        doc = fitz.open(stream=pdf_data, filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except Exception as e:
        print(f"Error extracting text from PDF {pdf_url}: {e}")
        return ""

def main():
    input_excel_path = r'C:\Users\Jaydeb\Downloads\ndb_all_projects_cards2.xlsx'  
    output_csv_path = r'C:\Users\Jaydeb\Downloads\ndb_projects_with_pdf_text.csv'

    # Load the Excel file
    df = pd.read_excel(input_excel_path)

    if 'Project URL' not in df.columns:
        print("Error: 'Project URL' column not found in Excel file.")
        return

    pdf_urls = []
    pdf_text_snippets = []

    total = len(df)
    for i, project_url in enumerate(df['Project URL'], 1):
        print(f"[{i}/{total}] Processing: {project_url}")

        pdf_link = get_pdf_link(project_url)
        pdf_urls.append(pdf_link if pdf_link else "")

        pdf_text = ""
        if pdf_link:
            pdf_text = extract_pdf_text(pdf_link)
        pdf_text_snippets.append(pdf_text[:1000] if pdf_text else "")

        time.sleep(1)  # polite delay between requests

    # Append new columns with PDF info
    df['PDF URL'] = pdf_urls
    df['PDF Text Snippet'] = pdf_text_snippets

    # Save output CSV
    df.to_csv(output_csv_path, index=False)
    print(f"Saved updated data with PDF snippets to {output_csv_path}")

if __name__ == "__main__":
    main()


[1/142] Processing: https://www.ndb.int/project/serentica-captive-renewable-energy-project/
[2/142] Processing: https://www.ndb.int/project/piramal-finance-affordable-housing-project/
[3/142] Processing: https://www.ndb.int/project/graca-aranha-silvania-energy-transmission-project/
[4/142] Processing: https://www.ndb.int/project/the-city-bank-sustainable-infrastructure-project/
[5/142] Processing: https://www.ndb.int/project/integration-social-and-sustainable-development-program-of-maceio/
[6/142] Processing: https://www.ndb.int/project/para-sanitation-development-project/
[7/142] Processing: https://www.ndb.int/project/hubei-huangshi-new-port-modern-logistics-hub-project/
[8/142] Processing: https://www.ndb.int/project/shriram-finance-sustainable-transport-project/
[9/142] Processing: https://www.ndb.int/project/brasilia-capital-of-solar-lighting-project/
[10/142] Processing: https://www.ndb.int/project/sael-300mw-renewable-energy-project/
[11/142] Processing: https://www.ndb.int/proj

In [ ]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import fitz  # pymupdf

# 1. Setup Chrome options to auto-download PDFs to folder "downloaded_pdfs"
download_dir = os.path.join(os.getcwd(), "downloaded_pdfs")
os.makedirs(download_dir, exist_ok=True)

chrome_options = Options()
prefs = {
    "plugins.always_open_pdf_externally": True,  # Disable PDF viewer; download PDFs instead
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
chrome_options.add_experimental_option("prefs", prefs)
# Optional: headless mode if you don't want browser UI
# chrome_options.add_argument("--headless")

driver = webdriver.Chrome(options=chrome_options)

def extract_text_from_pdf(file_path):
    try:
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except Exception as e:
        print(f"Failed to extract PDF text from {file_path}: {e}")
        return ""

def download_and_extract_pdf(pdf_url):
    print(f"Downloading PDF: {pdf_url}")
    driver.get(pdf_url)
    # Wait for download to complete (simple wait; better to check file existence)
    time.sleep(10)  # Adjust timing as needed based on file size and connection

    # Find the downloaded file in the folder (latest file)
    files = [os.path.join(download_dir, f) for f in os.listdir(download_dir)]
    if not files:
        print("No PDF file downloaded.")
        return ""

    latest_file = max(files, key=os.path.getctime)

    # Extract text
    text = extract_text_from_pdf(latest_file)
    return text

def main():
    input_csv = r'C:\Users\Jaydeb\Downloads\ndb_projects_with_pdf_text.csv'  # Your CSV path
    output_csv = r'C:\Users\Jaydeb\Downloads\ndb_projects_parsed_selenium.csv'

    df = pd.read_csv(input_csv)
    pdf_texts = []

    for i, pdf_url in enumerate(df['PDF URL'], 1):
        if pd.isna(pdf_url) or not pdf_url.strip():
            print(f"[{i}] Empty PDF URL, skipping.")
            pdf_texts.append("")
            continue

        text = download_and_extract_pdf(pdf_url)
        pdf_texts.append(text)
        print(f"[{i}] Extracted {len(text)} characters from PDF.")
        time.sleep(2)  # polite pause

    df['PDF Full Text'] = pdf_texts
    df.to_csv(output_csv, index=False)
    print(f"Saved all extracted PDF text to {output_csv}")

if __name__ == "__main__":
    main()
    driver.quit()


[1] Extracted 1518 characters from PDF.


In [1]:
import pandas as pd
import requests
import fitz  # PyMuPDF
import re
import time
import os

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"),
    "Referer": "https://www.ndb.int/",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "application/pdf,*/*;q=0.9"
}

def extract_pdf_text(pdf_url):
    try:
        response = requests.get(pdf_url, headers=HEADERS, timeout=20)
        response.raise_for_status()
        pdf_data = response.content
        doc = fitz.open(stream=pdf_data, filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except Exception as e:
        print(f"Error extracting text from PDF {pdf_url}: {e}")
        return ""

def parse_pdf_text_flexible(text):
    fields = [
        "Project Name",
        "Country",
        "Type",
        "Area of Operation",
        "Concept Approval Date",
        "Total Project Cost",
        "Proposed Limit of NDB Financing",
        "Borrower",
        "Project Entity",
        "Project Context",
        "Project Objective",
        "Project Description"
    ]
    
    parsed = {}
    for i, field in enumerate(fields):
        next_fields = fields[i+1:] if i+1 < len(fields) else []
        pattern = rf"{re.escape(field)}\s*(.*?)(?=" + "|".join([re.escape(f) for f in next_fields]) + "|$)"
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        parsed[field] = match.group(1).strip().replace('\n', ' ') if match else None
    return parsed

def main():
    input_csv = r'C:\Users\Jaydeb\Downloads\ndb_projects_with_pdf_text.csv'  # Adjust your CSV path here
    output_excel = r'C:\Users\Jaydeb\Downloads\ndb_projects_parsed.xlsx'

    if not os.path.isfile(input_csv):
        print(f"Input file not found: {input_csv}")
        return

    df = pd.read_csv(input_csv)

    if 'PDF URL' not in df.columns:
        print("Error: 'PDF URL' column missing in input CSV.")
        return

    parsed_rows = []
    total = len(df)

    for i, pdf_url in enumerate(df['PDF URL'], 1):
        if pd.isna(pdf_url) or not pdf_url.strip():
            print(f"[{i}/{total}] Skipping empty PDF URL")
            parsed_rows.append({field: None for field in [
                "Project Name","Country","Type","Area of Operation","Concept Approval Date",
                "Total Project Cost","Proposed Limit of NDB Financing","Borrower",
                "Project Entity","Project Context","Project Objective","Project Description"
            ]})
            continue
        
        print(f"[{i}/{total}] Processing PDF: {pdf_url}")
        text = extract_pdf_text(pdf_url)
        if text:
            parsed = parse_pdf_text_flexible(text)
        else:
            parsed = {field: None for field in [
                "Project Name","Country","Type","Area of Operation","Concept Approval Date",
                "Total Project Cost","Proposed Limit of NDB Financing","Borrower",
                "Project Entity","Project Context","Project Objective","Project Description"
            ]}
        parsed_rows.append(parsed)
        time.sleep(1)  # polite pause

    parsed_df = pd.DataFrame(parsed_rows)

    # Combine original data with parsed data
    final_df = pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

    # Save to Excel
    final_df.to_excel(output_excel, index=False)
    print(f"Saved parsed data to {output_excel}")

if __name__ == "__main__":
    main()


[1/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2025/06/Proposed-Projects_Summary_Public-Disclosure_Serentica.pdf
[2/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2025/05/Concept-Projects_PFL_Summary.pdf
[3/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2025/05/Proposed-Projects_Summary-for-Public-Disclosure_GATE-Concept-note.pdf
[4/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2024/04/CBL_Summary-for-Public-Disclosure.pdf
[5/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2025/04/Proposed-Projects_Summary-for-Public-Disclosure-Maceio-BRT-Project.pdf
[6/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2024/04/Para-Sanitation_Summary-for-Public-Disclosure.pdf
[7/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2025/02/Proposed-Projects_Summary-for-Public-Disclosure-Huangshi-Logistics.pdf
[8/142] Processing PDF: https://www.ndb.int/wp-content/uploads/2025/01/SFL_Summary-for-Public-Disclosure.pdf
[9/